# Bending playground

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, Angle, AltAz, ICRS, EarthLocation
from astropy.coordinates import BaseCoordinateFrame, BaseRepresentation, CartesianRepresentation, SphericalRepresentation, UnitSphericalRepresentation

from astropy.coordinates import RepresentationMapping
import astropy.coordinates.representation as r
from astropy.coordinates import TimeAttribute, QuantityAttribute, EarthLocationAttribute
from erfa import ufunc as erfa_ufunc
from astropy.utils import classproperty
from astropy.coordinates.matrix_utilities import rotation_matrix
from astropy.coordinates.baseframe import frame_transform_graph

from astropy.coordinates.transformations import FunctionTransform, StaticMatrixTransform

from astropy.modeling.rotations import SphericalRotationSequence

from scipy.interpolate import CloughTocher2DInterpolator, RBFInterpolator, SmoothBivariateSpline
import scipy
scipy.__version__

import dill as pickle

from datetime import datetime
from datetime import timezone

In [2]:

##6 parameter model

df = pd.read_csv("merged.csv")

# Convert once
az = np.deg2rad(df["Az"].values)
el = np.deg2rad(df["El"].values)

# --- Feature matrices ---
X_az = np.column_stack([
    np.ones_like(az),
    np.sin(az),
    np.cos(az),
])

X_el = np.column_stack([
    np.ones_like(el),
    np.sin(el),
    np.cos(el),
])

# Targets
y_az = df["Offset Az"].values
y_el = df["Offset El"].values

# --- Fit ---
coef_az, *_ = np.linalg.lstsq(X_az, y_az, rcond=None)
coef_el, *_ = np.linalg.lstsq(X_el, y_el, rcond=None)

# --- Feature builders ---
def az_features(az_deg):
    az = np.deg2rad(np.asarray(az_deg))
    return np.column_stack([
        np.ones_like(az),
        np.sin(az),
        np.cos(az),
    ])

def el_features(el_deg):
    el = np.deg2rad(np.asarray(el_deg))
    return np.column_stack([
        np.ones_like(el),
        np.sin(el),
        np.cos(el),
    ])

# --- Model functions (IMPORTANT FIX HERE) ---
def altfit_trig(el, **kwargs):
    X = el_features(el)
    return X @ coef_el

def azfit_trig(az, **kwargs):
    X = az_features(az)
    return X @ coef_az


"""
##10 parameter model
# Fit coefficients once from your CSV

df = pd.read_csv("merged.csv")

az = np.deg2rad(df["Az"].values)
el = np.deg2rad(df["El"].values)

X = np.column_stack([
    np.ones_like(az),
    np.sin(az),
    np.cos(az),
    np.sin(el),
    np.cos(el)
])

y_az = df["Offset Az"].values
y_el = df["Offset El"].values

# Solve least squares
coef_az, *_ = np.linalg.lstsq(X, y_az, rcond=None)
coef_el, *_ = np.linalg.lstsq(X, y_el, rcond=None)

def trig_features(az_deg, el_deg):
    az = np.deg2rad(az_deg)
    el = np.deg2rad(el_deg)

    return np.column_stack([
        np.ones_like(az),
        np.sin(az),
        np.cos(az),
        np.sin(el),
        np.cos(el)
    ])


def altfit_trig(az, el, **kwargs):
    X = trig_features(az, el)
    return X @ coef_el


def azfit_trig(az, el, **kwargs):
    X = trig_features(az, el)
    return X @ coef_az
"""

'\n##10 parameter model\n# Fit coefficients once from your CSV\n\ndf = pd.read_csv("merged.csv")\n\naz = np.deg2rad(df["Az"].values)\nel = np.deg2rad(df["El"].values)\n\nX = np.column_stack([\n    np.ones_like(az),\n    np.sin(az),\n    np.cos(az),\n    np.sin(el),\n    np.cos(el)\n])\n\ny_az = df["Offset Az"].values\ny_el = df["Offset El"].values\n\n# Solve least squares\ncoef_az, *_ = np.linalg.lstsq(X, y_az, rcond=None)\ncoef_el, *_ = np.linalg.lstsq(X, y_el, rcond=None)\n\ndef trig_features(az_deg, el_deg):\n    az = np.deg2rad(az_deg)\n    el = np.deg2rad(el_deg)\n\n    return np.column_stack([\n        np.ones_like(az),\n        np.sin(az),\n        np.cos(az),\n        np.sin(el),\n        np.cos(el)\n    ])\n\n\ndef altfit_trig(az, el, **kwargs):\n    X = trig_features(az, el)\n    return X @ coef_el\n\n\ndef azfit_trig(az, el, **kwargs):\n    X = trig_features(az, el)\n    return X @ coef_az\n'

In [3]:
class bending_model:
    def __init__(self, altfit_func, azfit_func, model_name, model_description):
        self.alt_func = altfit_func
        self.az_func = azfit_func
        self.name = model_name
        self.description = model_description

    def astro_to_drive(self, astro_alt, astro_az):
        deg_alt = Angle(astro_alt*u.deg).wrap_at(180*u.deg).deg
        deg_az = Angle(astro_az*u.deg).wrap_at(360*u.deg).deg
        drive_alt = (astro_alt-self.alt_func(deg_az, deg_alt, grid=False))
        drive_az = (astro_az-self.az_func(deg_az, deg_alt, grid=False))
        return (drive_alt, drive_az)

    def drive_to_astro(self, drive_alt, drive_az):
        deg_alt = Angle(drive_alt*u.deg).wrap_at(180*u.deg).deg
        deg_az = Angle(drive_az*u.deg).wrap_at(360*u.deg).deg
        astro_alt = (drive_alt+self.alt_func(deg_az, deg_alt, grid=False))
        astro_az = (drive_az+self.az_func(deg_az, deg_alt, grid=False))
        return (astro_alt, astro_az)

    def delta(self, alt, az):
        deg_alt = Angle(alt*u.deg).wrap_at(180*u.deg).deg
        deg_az = Angle(az*u.deg).wrap_at(360*u.deg).deg
        delta_alt = self.alt_func(deg_az, deg_alt, grid=False)
        delta_az = self.az_func(deg_az, deg_alt, grid=False)
        return (delta_alt, delta_az)

In [4]:
b = bending_model(
    altfit_func=altfit_trig,
    azfit_func=azfit_trig,
    model_name='testModel',
    model_description='blank'
)

"""
b = bending_model(altfit_func=altfit_spline,
                  azfit_func=azfit_spline,
                  model_name='test_model_v0',
                  model_description=f'generated on {datetime.now().strftime("%Y/%m/%d %H:%M:%S")} using toy data and scipy {scipy.__version__} SmoothBivariateSpline'
                 )
"""

'\nb = bending_model(altfit_func=altfit_spline,\n                  azfit_func=azfit_spline,\n                  model_name=\'test_model_v0\',\n                  model_description=f\'generated on {datetime.now().strftime("%Y/%m/%d %H:%M:%S")} using toy data and scipy {scipy.__version__} SmoothBivariateSpline\'\n                 )\n'

In [5]:
##Chi squared test
pred_alt = altfit_trig(df["El"].values)
pred_az  = azfit_trig(df["Az"].values)

df["Model Offset El"] = pred_alt
df["Model Offset Az"] = pred_az

df["El Residual"] = df["Offset El"] - df["Model Offset El"]
df["Az Residual"] = df["Offset Az"] - df["Model Offset Az"]

chi2_el = np.sum(df["El Residual"]**2)
chi2_az = np.sum(df["Az Residual"]**2)
chi2_total = chi2_el + chi2_az

##sigma here is calculated to be variance of the residuals; later, consider variance in...
##fitted parameters
##predictions

"""
N = len(df)
p = 3  #parameters per axis
sigma_el = np.sqrt(np.sum(df["El Residual"]**2) / (N - p))
sigma_az = np.sqrt(np.sum(df["Az Residual"]**2) / (N - p))
print("Estimated sigma El:", sigma_el)
print("Estimated sigma Az:", sigma_az)
"""

print("Total Chi^2:", chi2_total)
print("Chi^2 El:", chi2_el)
print("Chi^2 Az:", chi2_az)

print("El RMS:", np.sqrt(np.mean(df["El Residual"]**2)))
print("Az RMS:", np.sqrt(np.mean(df["Az Residual"]**2)))



Total Chi^2: 0.3225997961423951
Chi^2 El: 0.30724639108747964
Chi^2 Az: 0.015353405054915422
El RMS: 0.054616615335923456
Az RMS: 0.012209102546836883


In [6]:

"""
##10 parameter model
pred_alt = altfit_trig(df["Az"].values, df["El"].values)
pred_az  = azfit_trig(df["Az"].values, df["El"].values)

df["Model Offset El"] = pred_alt
df["Model Offset Az"] = pred_az

df["El Residual"] = df["Offset El"] - df["Model Offset El"]
df["Az Residual"]  = df["Offset Az"] - df["Model Offset Az"]

print("El RMS:", np.sqrt(np.mean(df["El Residual"]**2)))
print("Az RMS:", np.sqrt(np.mean(df["Az Residual"]**2)))

az_test = 292.689301  # degrees
el_test = 61.201706    # degrees

d_el = altfit_trig(az_test, el_test)
d_az = azfit_trig(az_test, el_test)

print("Predicted offsets:")
print("ΔAz:", d_az)
print("ΔEl:", d_el)
"""

##6 parameter model

#"""
for i, row in df.iterrows():
    pred_el = altfit_trig(row["El"])
    pred_az = azfit_trig(row["Az"])

    print("PrEl Offset:", pred_el, "PrAz Offset:", pred_az)
#"""

"""
pred_alt = altfit_trig(df["El"].values)
pred_az  = azfit_trig(df["Az"].values)

df["Model Offset El"] = pred_alt
df["Model Offset Az"] = pred_az

df["El Residual"] = df["Offset El"] - df["Model Offset El"]
df["Az Residual"] = df["Offset Az"] - df["Model Offset Az"]

print("El RMS:", np.sqrt(np.mean(df["El Residual"]**2)))
print("Az RMS:", np.sqrt(np.mean(df["Az Residual"]**2)))

az_test = 65.960297 # degrees
el_test = 67.259094   # degrees

d_el = altfit_trig(el_test)
d_az = azfit_trig(az_test)

print("Predicted offsets:")
print("ΔAz:", d_az)
print("ΔEl:", d_el)
"""



PrEl Offset: [-0.21950435] PrAz Offset: [0.05836365]
PrEl Offset: [-0.21898251] PrAz Offset: [0.05837427]
PrEl Offset: [-0.21862956] PrAz Offset: [0.05838146]
PrEl Offset: [-0.21749248] PrAz Offset: [0.05840406]
PrEl Offset: [-0.21569899] PrAz Offset: [0.05843172]
PrEl Offset: [-0.21444238] PrAz Offset: [0.05843619]
PrEl Offset: [-0.21248359] PrAz Offset: [0.05838417]
PrEl Offset: [-0.21165132] PrAz Offset: [0.05832133]
PrEl Offset: [-0.2103986] PrAz Offset: [0.05814256]
PrEl Offset: [-0.20807615] PrAz Offset: [0.05724359]
PrEl Offset: [-0.20659797] PrAz Offset: [0.0557392]
PrEl Offset: [-0.20536551] PrAz Offset: [0.05302354]
PrEl Offset: [-0.2042292] PrAz Offset: [0.0474519]
PrEl Offset: [-0.20322535] PrAz Offset: [0.03422554]
PrEl Offset: [-0.20286891] PrAz Offset: [0.02076413]
PrEl Offset: [-0.20277426] PrAz Offset: [0.00744445]
PrEl Offset: [-0.20295501] PrAz Offset: [-0.01407389]
PrEl Offset: [-0.20347577] PrAz Offset: [-0.03369339]
PrEl Offset: [-0.20424518] PrAz Offset: [-0.0491

'\npred_alt = altfit_trig(df["El"].values)\npred_az  = azfit_trig(df["Az"].values)\n\ndf["Model Offset El"] = pred_alt\ndf["Model Offset Az"] = pred_az\n\ndf["El Residual"] = df["Offset El"] - df["Model Offset El"]\ndf["Az Residual"] = df["Offset Az"] - df["Model Offset Az"]\n\nprint("El RMS:", np.sqrt(np.mean(df["El Residual"]**2)))\nprint("Az RMS:", np.sqrt(np.mean(df["Az Residual"]**2)))\n\naz_test = 65.960297 # degrees\nel_test = 67.259094   # degrees\n\nd_el = altfit_trig(el_test)\nd_az = azfit_trig(az_test)\n\nprint("Predicted offsets:")\nprint("ΔAz:", d_az)\nprint("ΔEl:", d_el)\n'